# 04 Advanced Models — Phase 1: Rigorous Ranking Evaluation

This notebook does **not** change notebooks 01–03. It re-scores the same three models
(Popularity, Item-CF, SVD) with a documented ranking protocol that also reports **NDCG@10**,
and runs a train/test **leakage check**.

**Protocol**
- Relevant: test rating ≥ 4.0
- Candidates: catalog movies the user did **not** rate in train
- Users evaluated: users with ≥ 1 relevant test item (and a train row)
- Metrics: Precision@10, Recall@10, NDCG@10 (plus RMSE for rating prediction)

In [1]:
import numpy as np
import pandas as pd
import utils

SEED = 42
np.random.seed(SEED)

train = pd.read_csv("data/train.csv")
test = pd.read_csv("data/test.csv")
M = utils.build_user_item(train)
print("matrix:", M.shape)
print("train rows:", len(train), "| test rows:", len(test))

matrix: (610, 2000)
train rows: 62711 | test rows: 15594


## 1. Toy NDCG sanity check

Binary gains, ranks start at 1. Ideal places all relevant items first.

In [2]:
# Ranking: relevant, miss, relevant  → gains [1, 0, 1]
dcg = utils._dcg_at_k([1, 0, 1], k=3)
idcg = utils._dcg_at_k([1, 1], k=3)  # two relevant items ideally first
ndcg = dcg / idcg
expected_dcg = 1.0 / np.log2(2) + 0.0 / np.log2(3) + 1.0 / np.log2(4)
expected_idcg = 1.0 / np.log2(2) + 1.0 / np.log2(3)
assert np.isclose(dcg, expected_dcg)
assert np.isclose(idcg, expected_idcg)
assert np.isclose(ndcg, expected_dcg / expected_idcg)
print(f"toy NDCG@3 = {ndcg:.6f}  (PASS)")

toy NDCG@3 = 0.919721  (PASS)


## 2. Leakage check

Confirm no (user, movie) pair is in both train and test, and that no test cell is
already filled in the train matrix that item means / SVD fit on.

In [3]:
leak = utils.leakage_check(train, test, M)
assert leak["passed"], "Leakage detected — stop and inspect the split."

leakage_check: PASS | overlap_pairs=0 | test_cells_filled_in_train_matrix=0


## 3. Fit the three existing models (same as notebook 03)

No retuning. SVD uses `random_state=42`.

In [4]:
pop = utils.PopularityBaseline().fit(M)
cf = utils.ItemBasedCF().fit(M)
svd = utils.SVDRecommender().fit(M, n_components=20, seed=SEED)
models = {"Popularity": pop, "Item-CF": cf, "SVD": svd}
print("fitted: popularity, item-CF, SVD(20 factors)")

fitted: popularity, item-CF, SVD(20 factors)


## 4. Evaluate with the Phase 1 ranking protocol

README baselines (from notebook 03, for side-by-side comparison):

| Model | RMSE ↓ | Precision@10 ↑ | Recall@10 ↑ |
|---|---|---|---|
| Popularity | 0.9381 | 0.0146 | 0.0119 |
| Item-based CF | 0.8298 | 0.0262 | 0.0107 |
| SVD (20 factors) | 0.8918 | 0.1293 | 0.1200 |

In [5]:
README = {
    "Popularity": {"RMSE": 0.9381, "P@10": 0.0146, "R@10": 0.0119},
    "Item-CF":    {"RMSE": 0.8298, "P@10": 0.0262, "R@10": 0.0107},
    "SVD":        {"RMSE": 0.8918, "P@10": 0.1293, "R@10": 0.1200},
}

rows = []
proto = None
for name, model in models.items():
    rmse = utils.rmse_on_test(model, test)
    rank = utils.ranking_metrics_at_k(model, M, test, k=10, rel_threshold=4.0)
    if proto is None:
        proto = rank
    rows.append({
        "model": name,
        "RMSE": round(rmse, 4),
        "P@10": round(rank["precision@k"], 4),
        "R@10": round(rank["recall@k"], 4),
        "NDCG@10": round(rank["ndcg@k"], 4),
        "n_users": rank["n_users_evaluated"],
        "README_RMSE": README[name]["RMSE"],
        "README_P@10": README[name]["P@10"],
        "README_R@10": README[name]["R@10"],
    })

results = pd.DataFrame(rows)
print("Evaluation protocol")
print(f"  rel_threshold = {proto['rel_threshold']}")
print("  candidates    = train_matrix columns minus movies the user rated in train")
print(f"  n_users_evaluated = {proto['n_users_evaluated']}")
print(f"  n_users_skipped   = {proto['n_users_skipped']}  (relevant test items but no train row)")
print()
display(results)

Evaluation protocol
  rel_threshold = 4.0
  candidates    = train_matrix columns minus movies the user rated in train
  n_users_evaluated = 591
  n_users_skipped   = 0  (relevant test items but no train row)



,model,RMSE,P@10,R@10,NDCG@10,n_users,README_RMSE,README_P@10,README_R@10
0,Popularity,0.9381,0.0146,0.0119,0.0148,591,0.9381,0.0146,0.0119
1,Item-CF,0.8298,0.0262,0.0107,0.0250,591,0.8298,0.0262,0.0107
2,SVD,0.8918,0.1293,0.1200,0.1755,591,0.8918,0.1293,0.1200


## 5. Notes on any README differences

If P@10 / R@10 match the README, the Phase 1 evaluator uses the same relevance threshold
and candidate set as `precision_recall_at_k` in notebook 03. NDCG@10 is new.
If numbers differ, the cause is one of: threshold, candidate set, or score-tie order —
document that here after running; do not rewrite the README in Phase 1.